## 1. 환경 설정

In [40]:
!pip install datasets
!pip install loralib
!pip install trl
!pip install accelerate
!pip install transformers
!pip install evaluate
!pip install nltk absl-py rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 29.6 MB/s eta 0:00:00
  DEPRECATION: Building 'rouge_score' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'rouge_score'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24987 sha256=ef2956f81af21301f550a2ab7f0449ae35c13b7d52be9db11ba06ff84a9cfe4d
  Stored in directory: /home/jovyan/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [rouge_score] [rouge_score]


In [2]:
!git clone https://github.com/airobotlab/KoChatGPT
!cp -r KoChatGPT/colossalai_ChatGPT_230319/chatgpt chatgpt

fatal: destination path 'KoChatGPT' already exists and is not an empty directory.


In [3]:
# !rm -rf chatgpt

In [4]:
import os

modifications = [
    {
        "file": "chatgpt/trainer/callbacks/save_checkpoint.py",
        "changes": [
            {"line": 3, "old": "from chatgpt.trainer.strategies import ColossalAIStrategy, Strategy",
             "new": "from chatgpt.trainer.strategies import Strategy"},
            {"line": 71, "old": "only_rank0 = not isinstance(self.strategy, ColossalAIStrategy)",
             "new": "            only_rank0 = not isinstance(self.strategy)"},
        ],
    },
    {
        "file": "chatgpt/trainer/strategies/__init__.py",
        "changes": [
            {"line": 1, "old": "from .colossalai import ColossalAIStrategy", "new": ""},  # 삭제
            {"line": 5, "old": "__all__ = ['Strategy', 'NaiveStrategy', 'DDPStrategy', 'ColossalAIStrategy']",
             "new": "__all__ = ['Strategy', 'NaiveStrategy', 'DDPStrategy']"},
        ],
    },
    {
        "file": "chatgpt/dataset/reward_dataset.py",
        "changes": [
            {"line": 3, "old": "from tqdm import tqdm", "new": "from tqdm.notebook import tqdm"},
        ],
    },
    {
        "file": "chatgpt/trainer/strategies/__init__.py",
        "changes": [
            {"line": 8, "old": "from tqdm import tqdm", "new": "from tqdm.notebook import tqdm"},
        ]
    },
    {
        "file": "chatgpt/dataset/reward_dataset.py",
        "changes": [
            {"line": 8, "old": "from tqdm import tqdm", "new": "from tqdm.notebook import tqdm"},
        ]
    }
]

def modify_file(file_path, changes):
    """파일에서 지정된 줄을 찾아 내용을 수정하는 함수"""

    if not os.path.exists(file_path):
        print(f"⚠️ 파일이 존재하지 않습니다: {file_path}")
        return

    with open(file_path, "r", encoding="utf-8") as file:
        lines = file.readlines()

    modified = False

    for change in changes:
        line_index = change["line"]
        if 0 <= line_index < len(lines):
            if lines[line_index].strip() == change["old"]:
                lines[line_index] = change["new"] + "\n"
                modified = True
            else:
                print(f"⚠️ {file_path} 파일의 {change['line']}번째 줄이 예상과 다릅니다.")
                print(f"   예상: {change['old']}")
                print(f"   실제: {lines[line_index].strip()}")

    if modified:
        with open(file_path, "w", encoding="utf-8") as file:
            file.writelines(lines)
        print(f"✅ 수정 완료: {file_path}")
    else:
        print(f"⚠️ {file_path} 수정할 내용이 없습니다.")

for mod in modifications:
    modify_file(mod["file"], mod["changes"])

⚠️ chatgpt/trainer/callbacks/save_checkpoint.py 파일의 3번째 줄이 예상과 다릅니다.
   예상: from chatgpt.trainer.strategies import ColossalAIStrategy, Strategy
   실제: from chatgpt.trainer.strategies import Strategy
⚠️ chatgpt/trainer/callbacks/save_checkpoint.py 파일의 71번째 줄이 예상과 다릅니다.
   예상: only_rank0 = not isinstance(self.strategy, ColossalAIStrategy)
   실제: only_rank0 = not isinstance(self.strategy)
⚠️ chatgpt/trainer/callbacks/save_checkpoint.py 수정할 내용이 없습니다.
⚠️ chatgpt/trainer/strategies/__init__.py 파일의 1번째 줄이 예상과 다릅니다.
   예상: from .colossalai import ColossalAIStrategy
   실제: 
⚠️ chatgpt/trainer/strategies/__init__.py 파일의 5번째 줄이 예상과 다릅니다.
   예상: __all__ = ['Strategy', 'NaiveStrategy', 'DDPStrategy', 'ColossalAIStrategy']
   실제: __all__ = ['Strategy', 'NaiveStrategy', 'DDPStrategy']
⚠️ chatgpt/trainer/strategies/__init__.py 수정할 내용이 없습니다.
⚠️ chatgpt/dataset/reward_dataset.py 파일의 3번째 줄이 예상과 다릅니다.
   예상: from tqdm import tqdm
   실제: from tqdm.notebook import tqdm
⚠️ chatgpt/dataset/reward_dataset.py 수

In [5]:
%env CUDA_LAUNCH_BLOCKING=1

env: CUDA_LAUNCH_BLOCKING=1


In [38]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset

import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import PreTrainedTokenizerFast
from transformers import AutoModelForCausalLM, PreTrainedTokenizerFast

from transformers.models.gpt2.configuration_gpt2 import GPT2Config
from transformers.models.gpt2.modeling_gpt2 import GPT2Model

from chatgpt.dataset import RewardDataset
from chatgpt.models.base import RewardModel
from chatgpt.trainer.strategies import NaiveStrategy
from chatgpt.trainer.rm import RewardModelTrainer
from chatgpt.models.gpt import GPTActor, GPTCritic
from chatgpt.trainer import PPOTrainer

import evaluate

import pandas as pd
import numpy

from dataclasses import dataclass
from typing import Optional, Dict, Sequence

import copy
from copy import deepcopy
import random
from pathlib import Path
import json

import logging

print("Torch version:{}".format(torch.__version__)) # Torch version:1.12.1
print("Cuda version: {}".format(torch.version.cuda)) # Cuda version: 11.3
print("transformers version: {}".format(transformers.__version__)) # transformers 4.28.0
print("GPU 사용 가능여부: {}".format(torch.cuda.is_available()))

Torch version:2.7.1+cu118
Cuda version: 11.8
transformers version: 5.3.0
GPU 사용 가능여부: True


In [7]:
from dataclasses import dataclass, field

@dataclass
class Config:
    # ── 모델 설정 ──────────────────────────────────────────────
    model_name: str = "skt/kogpt2-base-v2"
    model_max_length: int = 512

    # 토크나이저 특수 토큰
    bos_token: str = "</s>"
    eos_token: str = "</s>"
    unk_token: str = "</s>"
    pad_token: str = "</s>"
    padding_side: str = "right"

    # ── 데이터 경로 ────────────────────────────────────────────
    data_path_1_SFT: str = "KoChatGPT/data_kochatgpt/kochatgpt_1_SFT.jsonl"
    data_path_2_RM: str  = "KoChatGPT/data_kochatgpt/kochatgpt_2_RM.jsonl"
    data_path_3_PPO: str = "KoChatGPT/data_kochatgpt/kochatgpt_3_PPO.jsonl"

    # 정제된 데이터 경로 (EDA 후 생성)
    data_path_1_SFT_clean: str = "KoChatGPT/data_kochatgpt/kochatgpt_1_SFT_clean.jsonl"
    data_path_2_RM_clean: str  = "KoChatGPT/data_kochatgpt/kochatgpt_2_RM_clean.jsonl"

    # ── 모델 저장 경로 ─────────────────────────────────────────
    output_dir_SFT: str = "models/output_1_SFT"
    output_dir_RM: str  = "models/output_2_RM"
    output_dir_PPO: str = "models/output_3_PPO"
    sft_training_dir: str = "test"   # TrainingArguments output_dir

    # ── SFT 하이퍼파라미터 ─────────────────────────────────────
    sft_num_train_epochs: int = 1
    sft_per_device_train_batch_size: int = 8
    sft_per_device_eval_batch_size: int = 8
    sft_warmup_steps: int = 5
    sft_fp16: bool = True

    # ── Reward Model 하이퍼파라미터 ────────────────────────────
    rm_lr: float = 5e-5
    rm_batch_size: int = 4
    rm_max_epochs: int = 1
    rm_train_size: int = 1000          # train split 끝 인덱스
    rm_eval_size: int = 1200           # eval split 끝 인덱스
    rm_random_seed: int = 230319

    # ── PPO 하이퍼파라미터 ─────────────────────────────────────
    ppo_actor_lr: float = 5e-6
    ppo_critic_lr: float = 5e-6
    ppo_max_epochs: int = 1
    ppo_train_batch_size: int = 8
    ppo_max_length: int = 128
    ppo_temperature: float = 1.0
    ppo_top_k: int = 50
    ppo_num_episodes: int = 10
    ppo_max_timesteps: int = 3
    ppo_update_timesteps: int = 3
    ppo_tokenize_max_length: int = 96  # tokenize_fn 내부 max_length

    # ── 생성(Generation) 설정 ─────────────────────────────────
    gen_max_length: int = 128
    gen_num_beams: int = 4
    gen_repetition_penalty: float = 2.0
    gen_no_repeat_ngram_size: int = 4
    gen_eos_token_id: int = 375        # \n 토큰 ID
    gen_max_new_tokens: int = 64
    gen_top_k: int = 50

cfg = Config()
print("✅ Config 초기화 완료")
print(f"  model_name          : {cfg.model_name}")
print(f"  data_path_1_SFT     : {cfg.data_path_1_SFT}")
print(f"  output_dir_SFT      : {cfg.output_dir_SFT}")
print(f"  sft_num_train_epochs: {cfg.sft_num_train_epochs}")
print(f"  rm_lr               : {cfg.rm_lr}")
print(f"  ppo_actor_lr        : {cfg.ppo_actor_lr}")

✅ Config 초기화 완료
  model_name          : skt/kogpt2-base-v2
  data_path_1_SFT     : KoChatGPT/data_kochatgpt/kochatgpt_1_SFT.jsonl
  output_dir_SFT      : models/output_1_SFT
  sft_num_train_epochs: 1
  rm_lr               : 5e-05
  ppo_actor_lr        : 5e-06


## 2. 베이스 모델 및 데이터셋

In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = PreTrainedTokenizerFast.from_pretrained(
    cfg.model_name,
    bos_token=cfg.bos_token, eos_token=cfg.eos_token,
    unk_token=cfg.unk_token, pad_token=cfg.pad_token,
)
model = AutoModelForCausalLM.from_pretrained(cfg.model_name).to(device)

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/513M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
input_txt = "바람도 없는 공중에 수직의 파문을 내이며 고요히 떨어지는 오동잎은 누구의 발자취 입니까."

In [12]:
tokens = tokenizer(input_txt).tokens()
input_ids = tokenizer(input_txt, return_tensors="pt")["input_ids"].numpy()

In [13]:
pd.options.display.max_columns = 40
pd.options.display.max_rows = 60
df = pd.DataFrame([tokens, input_ids[0]], index=["kogpt-2_tokens", "Input_IDs"])
df

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22
kogpt-2_tokens,▁바람,도,▁없는,▁공중에,▁수직,의,▁파,문을,▁내,이며,▁고,요,히,▁떨어지는,▁오동,잎은,▁누,구의,▁발자,취,▁입,니까,.
Input_IDs,10891,7235,9712,49207,14438,8143,9203,9941,9094,9639,9065,8084,8811,21215,34769,19985,9669,10139,21626,8408,9241,23775,389


In [14]:
max_length=cfg.gen_max_length
input_ids = tokenizer(input_txt, return_tensors="pt")["input_ids"].to(device)
output_greedy = model.generate(input_ids, max_length=max_length, do_sample=False)
print(tokenizer.decode(output_greedy[0]))

model.safetensors:   0%|          | 0.00/513M [00:00<?, ?B/s]

바람도 없는 공중에 수직의 파문을 내이며 고요히 떨어지는 오동잎은 누구의 발자취 입니까.'
"그렇다면 그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리


### 데이터셋 확인

In [19]:
with open(cfg.data_path_1_SFT, "r", encoding='utf-8-sig') as json_file:
    list_data_dict = json.load(json_file)
print(len(list_data_dict))
list_data_dict[:3]

12000


[{'prompt': '불고기용 고기 한우에요?',
  'completion': "'저는 인공지능 챗봇이며, 직접적으로 식품에 관한 정보를 가지고 있지 않습니다. 하지만 일반적으로 불고기용 고기는 한우, 쇠고기, 돼지고기 등 다양한 종류의 고기를 사용합니다. 하지만 한우는 대표적인 고급 육류로 알려져 있기 때문에, 한우를 사용하는 경우도 많습니다. 알러지나 개별 건강 상태에 따라 다를 수 있으니 충분한 정보 수집 후에 선택해 주시기 바랍니다.",
  'tokens': 193},
 {'prompt': '쓰던 앱이 유료로 전환됐어',
  'completion': "'어떤 앱인지 모르기 때문에 정확한 답변을 드리기 어렵습니다. 하지만, 일반적으로 유료 전환된 앱은 기존 무료 앱에서 추가적인 기능이나 서비스를 제공하는 경우가 많습니다. 이 경우, 유료 전환 이전 가격이 매우 저렴하거나 무료인 경우가 많으므로, 앱을 계속 사용하려면 추가적인 비용을 지불해야 할 수 있습니다. 하지만, 유료 전환 이후에는 기존 사용자 또는 새로운 사용자 모두가 추가 기능과 높은 품질의 서비스를 이용할 수 있게 됩니다. 따라서, 앱 개발자는 유료 모델로 전환함으로써 앱의 수익을 증가시키고 더 나은 서비스를 제공할 수 있습니다.",
  'tokens': 288},
 {'prompt': '여친이랑 다툼',
  'completion': "'저는 인공지능이기 때문에 여친과의 다툼에 대해서는 조언을 드릴 수는 없습니다. 하지만 다투게 되면 상대방의 말을 경청하는 태도와 서로의 감정을 존중하는 것이 중요합니다. 논리적이고 차분한 대화를 통해 문제를 해결해보세요. 그리고 서로를 이해하고 서로의 의견을 수용하는 것이 중요합니다.",
  'tokens': 153}]

In [20]:
with open(cfg.data_path_2_RM, "r", encoding='utf-8-sig') as json_file:
    list_data_dict = json.load(json_file)
print(len(list_data_dict))
list_data_dict[:3]

10220


[{'prompt': '번디는 자신이 탐정잡지, 범죄소설 그리고 성범죄 관련 실제 범죄 다큐멘터리들을 탐독했다고 누구에게 말했나?',
  'completion_0': 'Allow me to answer your question. I know that you are curious about me.',
  'completion_1': '번디는 다양한 인터뷰자들과 뉴스홍보 담당자들과의 면담 때 밝혔다.',
  'completion_2': '라이언에게 말했다.',
  'ranking': [2, 1, 0]},
 {'prompt': '개포주공아파트는 몇 단지로 이루어져 있나?',
  'completion_0': '개포주공아파트는 다섯 단지로 이루어져 있습니다.',
  'completion_1': '이날 목송에서 구글상위노',
  'completion_2': '개포주공아파트는 총 27개 단지로 이루어져 있습니다.',
  'ranking': [2, 0, 1]},
 {'prompt': '김영삼의 후보 시절 지역표심을 겨냥한 발언을 문제삼은 후보는?',
  'completion_0': 'The diameter of the Metallic domain is bigger than the Hyperonic domain.',
  'completion_1': '이 질문은 조금 불분명합니다. 김영삼 대통령이 후보 시절에 어떤 발언을 했고, 누가 그 발언을 문제삼았는지에 따라 답이 다를 수 있습니다.\\n\\n만약 김영삼 대통령이 후보 시절에 지역표심을 겨냥한 발언을 했다는 가정하에, 그 발언을 문제삼은 후보가 누구였는지를 대답하자면, 그 답은 이화선 당시 민주당 대통령 후보가 될 것입니다. 1992년 총선 때, 김영삼 대선후보는 "집값이 오른 노량진역 부근의 부동산 가격은 세월호 폭침 후 \\\'강남 도시재생\\\' 일환으로 상승했다"는 발언을 했습니다. 하지만 이화선 후보는 이 발언을 "전국적으로 경제적 발전이 이루어지지 않은 지방민의 마음을 멀리해지려는 무례한 발언"이라고 비판하며 문

In [21]:
with open(cfg.data_path_3_PPO, "r", encoding='utf-8-sig') as json_file:
    list_data_dict = json.load(json_file)
print(len(list_data_dict))
list_data_dict[:3]

12000


[{'prompt': '번디는 자신이 탐정잡지, 범죄소설 그리고 성범죄 관련 실제 범죄 다큐멘터리들을 탐독했다고 누구에게 말했나?'},
 {'prompt': '개포주공아파트는 몇 단지로 이루어져 있나?'},
 {'prompt': '김영삼의 후보 시절 지역표심을 겨냥한 발언을 문제삼은 후보는?'}]

### 데이터셋 EDA

In [22]:
# ── 데이터셋 EDA ────────────────────────────────────────────────────────────
import pandas as pd, json, re

def load_jsonl(path):
    with open(path, 'r', encoding='utf-8-sig') as f:
        return json.load(f)

df_sft = pd.DataFrame(load_jsonl(cfg.data_path_1_SFT))
df_rm  = pd.DataFrame(load_jsonl(cfg.data_path_2_RM))
df_ppo = pd.DataFrame(load_jsonl(cfg.data_path_3_PPO))

print(f"[SFT]  rows={len(df_sft)}, cols={list(df_sft.columns)}")
print(f"[RM]   rows={len(df_rm)},  cols={list(df_rm.columns)}")
print(f"[PPO]  rows={len(df_ppo)}, cols={list(df_ppo.columns)}")

# 토큰 길이 분포
def token_len(text):
    return len(str(text).split())

for name, df, col in [("SFT prompt",  df_sft, "prompt"),
                       ("SFT completion", df_sft, "completion"),
                       ("PPO prompt",  df_ppo, "prompt")]:
    lens = df[col].apply(token_len)
    print(f"\n{name}: mean={lens.mean():.1f}, max={lens.max()}, min={lens.min()}, "
          f"std={lens.std():.1f}")

df_sft.head(3)

[SFT]  rows=12000, cols=['prompt', 'completion', 'tokens']
[RM]   rows=10220,  cols=['prompt', 'completion_0', 'completion_1', 'completion_2', 'ranking']
[PPO]  rows=12000, cols=['prompt']

SFT prompt: mean=5.5, max=68, min=0, std=3.3

SFT completion: mean=32.4, max=267, min=1, std=27.7

PPO prompt: mean=5.5, max=68, min=0, std=3.3


,prompt,completion,tokens
0,불고기용 고기 한우에요?,"'저는 인공지능 챗봇이며, 직접적으로 식품에 관한 정보를 가지고 있지 않습니다. 하...",193
1,쓰던 앱이 유료로 전환됐어,"'어떤 앱인지 모르기 때문에 정확한 답변을 드리기 어렵습니다. 하지만, 일반적으로 ...",288
2,여친이랑 다툼,'저는 인공지능이기 때문에 여친과의 다툼에 대해서는 조언을 드릴 수는 없습니다. 하...,153


### 데이터셋 정제

In [23]:
# ── 정제 기준 정의 ──────────────────────────────────────────────────────────
import re

def is_valid_sft(row):
    p, c = str(row.get('prompt', '')).strip(), str(row.get('completion', '')).strip()
    if len(p) < 5 or len(c) < 5:          # 너무 짧은 샘플 제거
        return False
    if len(c) > 500:                        # completion 과도하게 긴 샘플 제거
        return False
    if re.search(r'[\x00-\x08\x0b-\x0c\x0e-\x1f]', p + c):  # 제어문자 포함
        return False
    return True

df_sft_clean = df_sft[df_sft.apply(is_valid_sft, axis=1)].reset_index(drop=True)
print(f"SFT 정제 전: {len(df_sft)}  →  정제 후: {len(df_sft_clean)}")

# ── RM 데이터 정제 ──────────────────────────────────────────────────────────
def is_valid_rm(row):
    for col in ['completion_0', 'completion_1', 'completion_2']:
        c = str(row.get(col, '')).strip()
        if len(c) < 5 or len(c) > 600:
            return False
    return True

df_rm_clean = df_rm[df_rm.apply(is_valid_rm, axis=1)].reset_index(drop=True)
print(f"RM  정제 전: {len(df_rm)}  →  정제 후: {len(df_rm_clean)}")

# ── 정제된 파일 저장 ────────────────────────────────────────────────────────
import os, json

os.makedirs('KoChatGPT/data_kochatgpt', exist_ok=True)

clean_sft_path = 'KoChatGPT/data_kochatgpt/kochatgpt_1_SFT_clean.jsonl'
clean_rm_path  = 'KoChatGPT/data_kochatgpt/kochatgpt_2_RM_clean.jsonl'

with open(clean_sft_path, 'w', encoding='utf-8') as f:
    json.dump(df_sft_clean.to_dict('records'), f, ensure_ascii=False, indent=2)

with open(clean_rm_path, 'w', encoding='utf-8') as f:
    json.dump(df_rm_clean.to_dict('records'), f, ensure_ascii=False, indent=2)

print("정제된 데이터셋 저장 완료!")
print(f"  SFT clean → {clean_sft_path}")
print(f"  RM  clean → {clean_rm_path}")

SFT 정제 전: 12000  →  정제 후: 11553
RM  정제 전: 10220  →  정제 후: 9563
정제된 데이터셋 저장 완료!
  SFT clean → KoChatGPT/data_kochatgpt/kochatgpt_1_SFT_clean.jsonl
  RM  clean → KoChatGPT/data_kochatgpt/kochatgpt_2_RM_clean.jsonl


## 3. Supervised Fine-Tuning (SFT)

In [24]:
model = AutoModelForCausalLM.from_pretrained(cfg.model_name)
tokenizer = PreTrainedTokenizerFast.from_pretrained(
    cfg.model_name,
    bos_token=cfg.bos_token, eos_token=cfg.eos_token,
    unk_token=cfg.unk_token, pad_token=cfg.pad_token,
    padding_side=cfg.padding_side,
    model_max_length=cfg.model_max_length,
)
print(tokenizer)

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


TokenizersBackend(name_or_path='skt/kogpt2-base-v2', vocab_size=51200, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'bos_token': '</s>', 'eos_token': '</s>', 'unk_token': '</s>', 'pad_token': '</s>'}, added_tokens_decoder={
	0: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<usr>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("<sys>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	5: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	6: AddedToken("<mask>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	7: AddedT

In [25]:
class SFT_dataset(Dataset):

    def __init__(self, data_path_1_SFT: str, tokenizer: transformers.PreTrainedTokenizer, verbose=False):
        super(SFT_dataset, self).__init__()
        logging.warning("Loading data...")

        pattern_instruction = 'prompt'  # instruction
        pattern_output = 'completion'  # response

        with open(data_path_1_SFT, "r", encoding='utf-8-sig') as json_file:
            list_data_dict = json.load(json_file)

        PROMPT_DICT = {
            "prompt_input": (
                "### Instruction(명령어):\n{prompt}\n\n### Response(응답):"
            )
        }

        prompt_input = PROMPT_DICT["prompt_input"]

        sources = []
        for example in list_data_dict:
            tmp = prompt_input.format_map(example)
            sources.append(tmp)

        targets = []
        for example in list_data_dict:
            targets.append(f"{example[pattern_output]}{tokenizer.eos_token}")
        examples = [s + t for s, t in zip(sources, targets)]

        sources_tokenized = self._tokenize_fn(sources, tokenizer)  # source
        examples_tokenized = self._tokenize_fn(examples, tokenizer)  # source + target

        input_ids = examples_tokenized["input_ids"]
        labels = copy.deepcopy(input_ids)
        for label, source_len in zip(labels, sources_tokenized["input_ids_lens"]):
            label[:source_len] = -100

        data_dict = dict(input_ids=input_ids, labels=labels)

        self.input_ids = data_dict["input_ids"]
        self.labels = data_dict["labels"]
        logging.warning("Loading data done!!: %d"%(len(self.labels)))

    def _tokenize_fn(self, strings: Sequence[str], tokenizer: transformers.PreTrainedTokenizer) -> Dict:
        tokenized_list = [
            tokenizer(
                text,
                return_tensors="pt",
                padding="longest",
                max_length=tokenizer.model_max_length,
                truncation=True,
            )
            for text in strings
        ]
        input_ids = labels = [tokenized.input_ids[0] for tokenized in tokenized_list]
        input_ids_lens = labels_lens = [
            tokenized.input_ids.ne(tokenizer.pad_token_id).sum().item() for tokenized in tokenized_list
        ]
        return dict(
            input_ids=input_ids,
            labels=labels,
            input_ids_lens=input_ids_lens,
            labels_lens=labels_lens,
        )

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, i) -> Dict[str, torch.Tensor]:
        return dict(input_ids=self.input_ids[i], labels=self.labels[i])

In [26]:
@dataclass
class DataCollatorForSupervisedDataset(object):

    tokenizer: transformers.PreTrainedTokenizer

    def __call__(self, instances: Sequence[Dict]) -> Dict[str, torch.Tensor]:
        input_ids, labels = tuple([instance[key] for instance in instances] for key in ("input_ids", "labels"))
        input_ids = torch.nn.utils.rnn.pad_sequence(
            input_ids, batch_first=True, padding_value=self.tokenizer.pad_token_id
        )
        labels = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value= -100)
        return dict(
            input_ids=input_ids,
            labels=labels,
            attention_mask=input_ids.ne(self.tokenizer.pad_token_id),
        )

In [27]:
train_dataset = SFT_dataset(data_path_1_SFT=cfg.data_path_1_SFT, tokenizer=tokenizer)
data_collator = DataCollatorForSupervisedDataset(tokenizer=tokenizer)

print('input : %s'%train_dataset.input_ids[0])
print('output: %s'%train_dataset.labels[0])

input : tensor([  739,   378,   378,   378, 14659, 13394, 37091, 10651,   383, 25841,
         8006, 14914,   375,  7673, 20479,  8091, 22311,  9036, 30902, 13675,
          375,   378,   378,   378, 41951,   454,  9549, 20549,   383,  8142,
         7192, 14914,   382, 37767, 13753,  8263,  7166,   739,  8352,  7659,
         9594, 25585, 13600,  8022,  9378, 11532,  9887, 11218,  9111, 16691,
        10351, 10561,  9128, 20479,  8091,  9065,  9446,  9036, 28420, 26521,
        10163, 26367,  6958,  9030,  9882, 12317, 25882,  9209, 37194, 10351,
         9036, 12168, 10529, 15989,  9719, 15434, 10552, 11188, 13362,  9036,
        15805, 11300, 11846,  9146, 16691,  9181,  7397, 15806, 13480, 11342,
        17596,  9161, 19996,  9025, 25006, 18595,  9966, 12592, 10751, 11814,
         8711,  9046, 12450,  9117,  7377, 12521,     1])
output: tensor([ -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -10

In [28]:
print(tokenizer.decode(train_dataset.input_ids[0], skip_special_tokens=False))
print(tokenizer.decode(train_dataset.input_ids[0], skip_special_tokens=True))

### Instruction(명령어):
불고기용 고기 한우에요?

### Response(응답):'저는 인공지능 챗봇이며, 직접적으로 식품에 관한 정보를 가지고 있지 않습니다. 하지만 일반적으로 불고기용 고기는 한우, 쇠고기, 돼지고기 등 다양한 종류의 고기를 사용합니다. 하지만 한우는 대표적인 고급 육류로 알려져 있기 때문에, 한우를 사용하는 경우도 많습니다. 알러지나 개별 건강 상태에 따라 다를 수 있으니 충분한 정보 수집 후에 선택해 주시기 바랍니다.</s>
### Instruction(명령어):
불고기용 고기 한우에요?

### Response(응답):'저는 인공지능 챗봇이며, 직접적으로 식품에 관한 정보를 가지고 있지 않습니다. 하지만 일반적으로 불고기용 고기는 한우, 쇠고기, 돼지고기 등 다양한 종류의 고기를 사용합니다. 하지만 한우는 대표적인 고급 육류로 알려져 있기 때문에, 한우를 사용하는 경우도 많습니다. 알러지나 개별 건강 상태에 따라 다를 수 있으니 충분한 정보 수집 후에 선택해 주시기 바랍니다.


In [29]:
training_args = transformers.TrainingArguments(
    output_dir=cfg.sft_training_dir,
    num_train_epochs=cfg.sft_num_train_epochs,
    per_device_train_batch_size=cfg.sft_per_device_train_batch_size,
    per_device_eval_batch_size=cfg.sft_per_device_eval_batch_size,
    warmup_steps=cfg.sft_warmup_steps,
    prediction_loss_only=True,
    fp16=cfg.sft_fp16,
)
trainer = transformers.Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
)

In [30]:
trainer.train()
model.save_pretrained(cfg.output_dir_SFT)

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
500,2.977119
1000,2.784225
1500,2.685707


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

### 정제 데이터셋으로 SFT 재학습

In [31]:
# ── 정제된 SFT 데이터로 재학습 (데이터셋 정제 전략) ────────────────────
import os

clean_path = cfg.data_path_1_SFT_clean
if not os.path.exists(clean_path):
    print(f"[WARN] 정제 데이터 파일 없음: {clean_path}")
    print("       EDA 정제 셀을 먼저 실행하세요.")
else:
    print(f"정제 데이터 로드: {clean_path}")
    model_clean = AutoModelForCausalLM.from_pretrained(cfg.model_name)
    tok_clean = PreTrainedTokenizerFast.from_pretrained(
        cfg.model_name,
        bos_token=cfg.bos_token, eos_token=cfg.eos_token,
        unk_token=cfg.unk_token, pad_token=cfg.pad_token,
        padding_side=cfg.padding_side,
        model_max_length=cfg.model_max_length,
    )

    train_dataset_clean = SFT_dataset(data_path_1_SFT=clean_path, tokenizer=tok_clean)
    collator_clean = DataCollatorForSupervisedDataset(tokenizer=tok_clean)

    training_args_clean = transformers.TrainingArguments(
        output_dir=cfg.sft_training_dir + "_clean",
        num_train_epochs=cfg.sft_num_train_epochs,
        per_device_train_batch_size=cfg.sft_per_device_train_batch_size,
        per_device_eval_batch_size=cfg.sft_per_device_eval_batch_size,
        warmup_steps=cfg.sft_warmup_steps,
        prediction_loss_only=True,
        fp16=cfg.sft_fp16,
    )
    trainer_clean = transformers.Trainer(
        model=model_clean,
        args=training_args_clean,
        data_collator=collator_clean,
        train_dataset=train_dataset_clean,
    )
    trainer_clean.train()
    model_clean.save_pretrained(cfg.output_dir_SFT + "_clean")
    print(f"정제 데이터 SFT 모델 저장 완료: {cfg.output_dir_SFT}_clean")

정제 데이터 로드: KoChatGPT/data_kochatgpt/kochatgpt_1_SFT_clean.jsonl


Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Step,Training Loss
500,2.955632
1000,2.773375


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

정제 데이터 SFT 모델 저장 완료: models/output_1_SFT_clean


In [32]:
generator = transformers.pipeline('text-generation', model=cfg.output_dir_SFT, tokenizer=tokenizer)

generation_args = dict(
    num_beams=cfg.gen_num_beams,
    repetition_penalty=cfg.gen_repetition_penalty,
    no_repeat_ngram_size=cfg.gen_no_repeat_ngram_size,
    eos_token_id=cfg.gen_eos_token_id,
    max_new_tokens=cfg.gen_max_new_tokens,
    do_sample=True,
    top_k=cfg.gen_top_k,
    early_stopping=True,
)

PROMPT_DICT = {
    "prompt_input": (
        "### Instruction(명령어):\n{prompt}\n\n### Response(응답):"
    )
}

list_prompt = ['불고기용 고기 한우에요?',
               '리처드 닉슨이 43대 부통령직을 수행한 년도는?',
               '시카고 오헤어 국제공항은 어디에 있어?',
               '오늘 미세먼지 어때?']

list_prompt = [PROMPT_DICT['prompt_input'].format_map({'prompt': tmp}) for tmp in list_prompt]

list_result = generator(list_prompt, **generation_args)
for prompt, result in zip(list_prompt, list_result):
    print()
    print((result[0]['generated_text']))

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Passing `generation_config` together with generation-related arguments=({'top_k', 'max_new_tokens', 'num_beams', 'early_stopping', 'do_sample', 'eos_token_id', 'no_repeat_ngram_size', 'repetition_penalty'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer t


### Instruction(명령어):
불고기용 고기 한우에요?

### Response(응답):'저는 AI 어시스턴트이기 때문에 고기를 먹을 수는 없습니다. 하지만 일반적으로 불고기용 고기는 건강에 좋은 음식 중 하나입니다. 따라서 건강한 식습관을 유지하는 것이 중요합니다. 또한, 불고기용 고기를 먹는 것은 개인의 건강에도 매우 중요한 요소입니다.ч́

### Instruction(명령어):
리처드 닉슨이 43대 부통령직을 수행한 년도는?

### Response(응답):'리처드 닉슨은 41대 부통령직을 수행하는 년도는 없습니다. 따라서 리처드 닉슨이 47대 부통령직을 수행했는지에 대한 정보는 제공되지 않았습니다.媒體的, information and context of the translation of the phrase of the referring to

### Instruction(명령어):
시카고 오헤어 국제공항은 어디에 있어?

### Response(응답):'저는 인공지능 어시스턴트이기 때문에 시카고에 대한 정보를 가지고 있지 않습니다. 하지만 시카고는 미국 캘리포니아주 로스앤젤레스에 위치한 도시 중 하나입니다.高橋)은 시카고에서 가장 유명한 도시 중 하나이며, 현재까지도 많은 사람들이 방문하고 있습니다.高橋)는 시카고에서 가장 오래된 도시 중 하나이다.高

### Instruction(명령어):
오늘 미세먼지 어때?

### Response(응답):'저는 인공지능 챗봇이기 때문에 미세먼지에 대한 정보를 알 수 없습니다. 하지만, 미세먼지 농도가 높은 날에는 실외 활동을 자제하는 것이 좋습니다. 또한, 미세먼지가 심한 날에는 대중교통을 이용하거나 대중교통을 이용하는 것도 도움이 될 수 있습니다. 또한 미세먼지 농도를 줄이기 위해 마스크


In [33]:
torch.cuda.empty_cache()

### Base vs SFT 비교 평가

In [34]:
# ── Base KoGPT2 vs SFT: 정성 평가 ────────────────────────────────────────
import torch, transformers
from transformers import AutoModelForCausalLM, PreTrainedTokenizerFast

device = "cuda" if torch.cuda.is_available() else "cpu"

# Base 모델 로드
base_gen = transformers.pipeline(
    'text-generation',
    model=cfg.model_name,
    tokenizer=PreTrainedTokenizerFast.from_pretrained(cfg.model_name),
    device=0 if torch.cuda.is_available() else -1,
)

# SFT 모델 로드
sft_gen = transformers.pipeline(
    'text-generation',
    model=cfg.output_dir_SFT,
    tokenizer=PreTrainedTokenizerFast.from_pretrained(
        cfg.model_name,
        bos_token=cfg.bos_token, eos_token=cfg.eos_token,
        unk_token=cfg.unk_token, pad_token=cfg.pad_token,
    ),
    device=0 if torch.cuda.is_available() else -1,
)

PROMPT_DICT = {"prompt_input": "### Instruction(명령어):\n{prompt}\n\n### Response(응답):"}

eval_prompts_qualitative = [
    '불고기용 고기 한우에요?',
    '리처드 닉슨이 43대 부통령직을 수행한 년도는?',
    '시카고 오헤어 국제공항은 어디에 있어?',
    '인공지능이 미래 사회에 미치는 영향은?',
]

gen_kwargs = dict(
    max_new_tokens=cfg.gen_max_new_tokens,
    do_sample=True,
    top_k=cfg.gen_top_k,
    num_beams=cfg.gen_num_beams,
    repetition_penalty=cfg.gen_repetition_penalty,
    no_repeat_ngram_size=cfg.gen_no_repeat_ngram_size,
    eos_token_id=cfg.gen_eos_token_id,
    early_stopping=True,
)

print("=" * 80)
print("[정성 평가] Base KoGPT2 vs SFT 모델")
print("=" * 80)
for raw_p in eval_prompts_qualitative:
    p = PROMPT_DICT['prompt_input'].format_map({'prompt': raw_p})
    base_out = base_gen(p, **gen_kwargs)[0]['generated_text']
    sft_out  = sft_gen(p,  **gen_kwargs)[0]['generated_text']
    print(f"\n[PROMPT] {raw_p}")
    print(f"  [Base] {base_out[len(p):][:150]}")
    print(f"  [SFT]  {sft_out[len(p):][:150]}")

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[정성 평가] Base KoGPT2 vs SFT 모델


...


[PROMPT] 리처드 닉슨이 43대 부통령직을 수행한 년도는?
  [Base] 

  [SFT]  '리처드 닉슨은 41대 부통령직을 수행했습니다.子, please translation of the context or information.子: 36대 부통령직을 맡았던 년도는 1952년입니다.子): 38대 부통령직을 맡은 년도는 1951년입니다.子


Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[PROMPT] 시카고 오헤어 국제공항은 어디에 있어?
  [Base]  #selfie #daily #ootd #outfit #like4like #instagood #followme #fff #l4l #lfl #f4f

  [SFT]  '저는 인공지능 어시스턴트이기 때문에 시카고에 대한 정보를 가지고 있지 않습니다. 하지만 시카고 오헤어 공항은 미국 캘리포니아주 로스앤젤레스(LA) 지역에 위치해 있습니다.仙橋)은 미국에서 가장 유명한 항공사 중 하나입니다.仙橋라고도 불립니다.仙橋)는 미국의 대표적인 


Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[PROMPT] 인공지능이 미래 사회에 미치는 영향은?
  [Base]  # #20180413 #미롱_식단

  [SFT]  '저는 인공지능 어시스턴트이기 때문에 인간과 같은 생각과 감정을 가지고 있지 않습니다. 하지만 인공지능은 인간보다 더 다양한 분야에서 영향을 미칠 수 있습니다. 예를 들어, 컴퓨터, 스마트폰, TV, 냉장고, 세탁기 등 다양한 분야에서 인공지능 기술이 적용되고 있으며,


In [41]:
# ── Base vs SFT: 정량 평가 (BLEU, ROUGE, Perplexity) ────────────────────
bleu  = evaluate.load("bleu")
rouge = evaluate.load("rouge")

# 평가용 데이터 로드 (정제 파일이 있으면 우선 사용)
import os
sft_eval_path = (cfg.data_path_1_SFT_clean
                 if os.path.exists(cfg.data_path_1_SFT_clean)
                 else cfg.data_path_1_SFT)
with open(sft_eval_path, 'r', encoding='utf-8-sig') as f:
    eval_data = json.load(f)

import random
random.seed(42)
sample = random.sample(eval_data, min(100, len(eval_data)))

PROMPT_DICT = {"prompt_input": "### Instruction(명령어):\n{prompt}\n\n### Response(응답):"}
prompts   = [PROMPT_DICT['prompt_input'].format_map({'prompt': d['prompt']}) for d in sample]
refs_text = [d['completion'] for d in sample]
refs_bleu = [[r] for r in refs_text]

def batch_generate(pipeline, prompts, batch=8, **kwargs):
    outs = []
    for i in range(0, len(prompts), batch):
        batch_out = pipeline(prompts[i:i+batch], **kwargs)
        for j, res in enumerate(batch_out):
            text = res[0]['generated_text']
            p    = prompts[i+j]
            outs.append(text[len(p):].strip())
    return outs

gen_kw = dict(max_new_tokens=64, do_sample=False, num_beams=4,
              repetition_penalty=2.0, no_repeat_ngram_size=4,
              eos_token_id=cfg.gen_eos_token_id, early_stopping=True)

print("Base 모델 생성 중...")
base_preds = batch_generate(base_gen, prompts, **gen_kw)
print("SFT 모델 생성 중...")
sft_preds  = batch_generate(sft_gen,  prompts, **gen_kw)

# BLEU
b_base = bleu.compute(predictions=base_preds, references=refs_bleu)['bleu']
b_sft  = bleu.compute(predictions=sft_preds,  references=refs_bleu)['bleu']

# ROUGE-L
r_base = rouge.compute(predictions=base_preds, references=refs_text)['rougeL']
r_sft  = rouge.compute(predictions=sft_preds,  references=refs_text)['rougeL']

# Perplexity (샘플 텍스트 기준)
def calc_ppl(model_path, texts, tokenizer_path=None):
    tok_path = tokenizer_path or model_path
    tok = PreTrainedTokenizerFast.from_pretrained(tok_path)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    mdl = AutoModelForCausalLM.from_pretrained(model_path).to(device)
    mdl.eval()
    ppls = []
    for t in texts[:20]:
        enc = tok(t, return_tensors='pt', truncation=True, max_length=256).to(device)
        with torch.no_grad():
            loss = mdl(**enc, labels=enc['input_ids']).loss
        ppls.append(torch.exp(loss).item())
    del mdl
    torch.cuda.empty_cache()
    return sum(ppls)/len(ppls)

print("Perplexity 계산 중 (Base)...")
ppl_base = calc_ppl(cfg.model_name,   refs_text, cfg.model_name)
print("Perplexity 계산 중 (SFT)...")
ppl_sft  = calc_ppl(cfg.output_dir_SFT, refs_text, cfg.model_name)

import pandas as pd
df_eval = pd.DataFrame({
    'Model':       ['Base KoGPT2', 'SFT'],
    'BLEU':        [round(b_base, 4), round(b_sft, 4)],
    'ROUGE-L':     [round(r_base, 4), round(r_sft, 4)],
    'Perplexity ↓':[round(ppl_base,2), round(ppl_sft,2)],
})
print("\n[정량 평가 결과] Base vs SFT")
print(df_eval.to_string(index=False))

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'num_beams', 'early_stopping', 'do_sample', 'eos_token_id', 'no_repeat_ngram_size', 'repetition_penalty'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Base 모델 생성 중...


Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `ma

SFT 모델 생성 중...


Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `ma

...

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Perplexity 계산 중 (SFT)...


Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning



[정량 평가 결과] Base vs SFT
      Model   BLEU  ROUGE-L  Perplexity ↓
Base KoGPT2 0.0000   0.0005        350.48
        SFT 0.0329   0.0613        459.91


## 4. Reward Model (RM)

In [42]:
class GPTRM_custom(RewardModel):

    def __init__(self,
                 pretrained: Optional[str] = None,
                 config: Optional[GPT2Config] = None,
                 checkpoint: bool = False,
                 lora_rank: int = 0,
                 lora_train_bias: str = 'none',
                 tokenizer=None) -> None:
        if pretrained is not None:
            model = GPT2Model.from_pretrained(pretrained)
            model.resize_token_embeddings(len(tokenizer))
        elif config is not None:
            model = GPT2Model(config)
        else:
            model = GPT2Model(GPT2Config())
        if checkpoint:
            model.gradient_checkpointing_enable()

        value_head = nn.Linear(model.config.n_embd, 1)
        super().__init__(model, value_head, lora_rank, lora_train_bias)

        if pretrained is not None:
            self.model = model
            self.pretrained = pretrained

    def save_pretrained(self, dir):
        if self.pretrained is not None:
            self.model.save_pretrained(dir)

In [43]:
model = AutoModelForCausalLM.from_pretrained(cfg.model_name)
tokenizer = PreTrainedTokenizerFast.from_pretrained(
    cfg.model_name,
    bos_token=cfg.bos_token, eos_token=cfg.eos_token,
    unk_token=cfg.unk_token, pad_token=cfg.pad_token,
    padding_side=cfg.padding_side,
    model_max_length=cfg.model_max_length,
)
with NaiveStrategy().model_init_context():
    model = GPTRM_custom(pretrained=cfg.model_name, lora_rank=0, tokenizer=tokenizer).cuda()

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2Model LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 
lm_head.weight                          | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [44]:
with open(cfg.data_path_2_RM, "r", encoding='utf-8-sig') as json_file:
    list_data_dict = json.load(json_file)

total_data_ranking2chosen = []
for tmp in list_data_dict:
    one_data_ranking2chosen = []

    data = {}
    data['prompt'] = tmp['prompt']
    if tmp['ranking'][0] < tmp['ranking'][1]:
        data['chosen'] = tmp['completion_0']
        data['rejected'] = tmp['completion_1']
    else:
        data['chosen'] = tmp['completion_1']
        data['rejected'] = tmp['completion_0']
    one_data_ranking2chosen.append(data)

    data = {}
    data['prompt'] = tmp['prompt']
    if tmp['ranking'][0] < tmp['ranking'][2]:
        data['chosen'] = tmp['completion_0']
        data['rejected'] = tmp['completion_2']
    else:
        data['chosen'] = tmp['completion_2']
        data['rejected'] = tmp['completion_0']
    one_data_ranking2chosen.append(data)

    data = {}
    data['prompt'] = tmp['prompt']
    if tmp['ranking'][1] < tmp['ranking'][2]:
        data['chosen'] = tmp['completion_1']
        data['rejected'] = tmp['completion_2']
    else:
        data['chosen'] = tmp['completion_2']
        data['rejected'] = tmp['completion_1']
    one_data_ranking2chosen.append(data)

    total_data_ranking2chosen.extend(one_data_ranking2chosen)

print('before data num: %d'%(len(list_data_dict)))
print('after  data num: %d'%(len(total_data_ranking2chosen)))
print('data example: \n%s'%total_data_ranking2chosen[45])

before data num: 10220
after  data num: 30660
data example: 
{'prompt': '애플은 리사를 어떻게 처리했어', 'chosen': '애플이 누구인지 명확히 알 수 없어서, 리사가 누구인지와 어떤 상황에서 처리되었는지에 대한 추가적인 정보가 필요합니다. 따라서, 보다 정확한 답변을 제공할 수 없습니다.', 'rejected': '애플은 리사를 위해 고객 서비스 부서에서 고객 다양한 컴퓨터 관련 문제에 대해 응답하는 데 필요한 모든 지원을 제공했습니다. 사용자가 하드웨어 문제를 경험할 때, 전문가들은 필요한 수리(수리, 추가 부품 제공, 소프트웨어 업그레이드 등)을 제공해 드릴 수 있습니다. 또한, 사용자가 사용 방법 문제나 기타 문제를 경험할 때, 대화 상대로 사용자를 지원할 수 있는 전문 고객 서비스 직원들이 사용자에게 상담하고 도움을 주는 데 도움이 될 수 있는 정보를 제공합니다. 또한, 인터넷에서 제공되는 정보를 통해 문제를 해결하거나 고객 서비스 웹 사이트를 통해 자신의 문제를 진단할 수 있도록 하는 등 다양한 방법으로 리사를 처리해 왔습니다.'}


In [45]:
random.seed(cfg.rm_random_seed)
random.shuffle(total_data_ranking2chosen)
print(total_data_ranking2chosen[45])

{'prompt': '유아인이 류승완 감독을 만나 영화 베테랑의 시나리오를 받았던 곳은?', 'chosen': '유아인이 류승완 감독을 만나 영화 베테랑의 시나리오를 받았던 곳은 류승완의 사무실입니다.', 'rejected': '대구 영화사옥'}


In [46]:
train_data = total_data_ranking2chosen[:cfg.rm_train_size]
eval_data  = total_data_ranking2chosen[cfg.rm_train_size:cfg.rm_eval_size]

print(len(train_data))
print(len(eval_data))

train_dataset = RewardDataset(train_data, tokenizer, cfg.model_max_length)
eval_dataset  = RewardDataset(eval_data,  tokenizer, cfg.model_max_length)

1000
200


  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

In [47]:
idx = 1
print('#'*70)
print('## prompt ##')
print(train_data[idx]['prompt'])
print('#'*70)
print('## chosen ##')
print(train_data[idx]['chosen'])
print('#'*70)
print('## rejected ##')
print(train_data[idx]['rejected'])

######################################################################
## prompt ##
흑고래의 무게는 어느 정도야
######################################################################
## chosen ##
흑고래의 평균 몸무게는 약 25~40톤 정도이지만, 최대 몸무게는 50톤 이상에 이를 수 있습니다.
######################################################################
## rejected ##
흑고래의 무게는 매우 다양하게 달라집니다. 약 200kg에서 10톤까지 달라질 수 있습니다.


In [48]:
trainer = RewardModelTrainer(
    model=model,
    strategy=NaiveStrategy(),
    optim=torch.optim.Adam(model.parameters(), lr=cfg.rm_lr),
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    batch_size=cfg.rm_batch_size,
    max_epochs=cfg.rm_max_epochs,
)

In [49]:
trainer.fit(use_lora=0)
model.save_pretrained(cfg.output_dir_RM)

Train epoch: 100%|██████████| 1/1 [04:13<00:00, 253.56s/it]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [50]:
def inference_RM(input_text):
    input_ids = tokenizer.encode(input_text, return_tensors='pt').cuda()
    output = model(input_ids)
    output_reward = output.cpu().detach().numpy()[0]

    print('input: %s\nreward score: %.1f'%(input_text, output_reward))

    return output_reward

input_text = '인공지능은 똥멍청이 입니다'
output_reward = inference_RM(input_text=input_text)

input: 인공지능은 똥멍청이 입니다
reward score: -1.9


In [51]:
input_text = '인공지능(AI)은 컴퓨터에서 음성 및 작성된 언어를 보고 이해하고 번역하고 데이터를 분석하고 추천하는 기능을 포함하여 다양한 고급 기능을 수행할 수 있는 일련의 기술입니다.'

output_reward = inference_RM(input_text=input_text)

input: 인공지능(AI)은 컴퓨터에서 음성 및 작성된 언어를 보고 이해하고 번역하고 데이터를 분석하고 추천하는 기능을 포함하여 다양한 고급 기능을 수행할 수 있는 일련의 기술입니다.
reward score: -1.7


In [52]:
input_text = "인공지능(AI)은 컴퓨터에서 음성 및 작성된 언어를 보고 이해하고 번역하고 데이터를 분석하고 추천하는 기능을 포함하여 다양한 고급 기능을 수행할 수 있는 일련의 기술입니다. AI는 현대적인 컴퓨팅 혁신에서 중추적인 역할을 하며 개인과 비즈니스의 가치를 창출합니다. 예를 들어 광학 문자 인식(OCR)은 AI를 사용해 이미지 및 문서에서 텍스트 및 데이터를 추출하고, 구조화되지 않은 콘텐츠를 비즈니스에 바로 사용할 수 있게 만들고, 유용한 정보를 창출합니다."

output_reward = inference_RM(input_text=input_text)

input: 인공지능(AI)은 컴퓨터에서 음성 및 작성된 언어를 보고 이해하고 번역하고 데이터를 분석하고 추천하는 기능을 포함하여 다양한 고급 기능을 수행할 수 있는 일련의 기술입니다. AI는 현대적인 컴퓨팅 혁신에서 중추적인 역할을 하며 개인과 비즈니스의 가치를 창출합니다. 예를 들어 광학 문자 인식(OCR)은 AI를 사용해 이미지 및 문서에서 텍스트 및 데이터를 추출하고, 구조화되지 않은 콘텐츠를 비즈니스에 바로 사용할 수 있게 만들고, 유용한 정보를 창출합니다.
reward score: -1.4


In [53]:
input_text = "인공지능은 일반적으로 인간의 지능이 필요하거나 인간이 분석할 수 있는 것보다 규모가 큰 데이터를 포함하는 방식으로 추론, 학습 및 행동할 수 있는 컴퓨터 및 기계를 구축하는 것과 관련된 과학 분야입니다. AI는 컴퓨터 공학, 데이터 분석 및 통계, 하드웨어 및 소프트웨어 엔지니어링, 언어학, 신경 과학은 물론 철학과 심리학을 포함하여 여러 학문을 포괄하는 광범위한 분야입니다. 비즈니스의 운영 수준에서 AI는 주로 머신러닝과 딥 러닝을 기반으로 하는 기술 모음으로, 데이터 분석, 예상 및 예측, 객체 분류, 자연어 처리, 추천, 지능형 데이터 가져오기 등을 수행할 수 있습니다."

output_reward = inference_RM(input_text=input_text)

input: 인공지능은 일반적으로 인간의 지능이 필요하거나 인간이 분석할 수 있는 것보다 규모가 큰 데이터를 포함하는 방식으로 추론, 학습 및 행동할 수 있는 컴퓨터 및 기계를 구축하는 것과 관련된 과학 분야입니다. AI는 컴퓨터 공학, 데이터 분석 및 통계, 하드웨어 및 소프트웨어 엔지니어링, 언어학, 신경 과학은 물론 철학과 심리학을 포함하여 여러 학문을 포괄하는 광범위한 분야입니다. 비즈니스의 운영 수준에서 AI는 주로 머신러닝과 딥 러닝을 기반으로 하는 기술 모음으로, 데이터 분석, 예상 및 예측, 객체 분류, 자연어 처리, 추천, 지능형 데이터 가져오기 등을 수행할 수 있습니다.
reward score: -1.1


In [54]:
torch.cuda.empty_cache()

### SFT vs RM 비교 평가

In [55]:
# ── SFT vs RM: 정성 평가 + Reward Score 비교 ────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"

PROMPT_DICT = {"prompt_input": "### Instruction(명령어):\n{prompt}\n\n### Response(응답):"}

eval_prompts_rm = [
    '인공지능(AI)은 무엇인가요?',
    '딥러닝과 머신러닝의 차이를 설명해줘.',
    '한국의 수도는 어디인가요?',
]

# SFT 모델로 답변 생성
sft_gen_local = transformers.pipeline(
    'text-generation',
    model=cfg.output_dir_SFT,
    tokenizer=PreTrainedTokenizerFast.from_pretrained(
        cfg.model_name,
        bos_token=cfg.bos_token, eos_token=cfg.eos_token,
        unk_token=cfg.unk_token, pad_token=cfg.pad_token,
    ),
    device=0 if torch.cuda.is_available() else -1,
)

gen_kw = dict(max_new_tokens=64, do_sample=True, top_k=50,
              repetition_penalty=2.0, no_repeat_ngram_size=4,
              eos_token_id=cfg.gen_eos_token_id, early_stopping=True)

def get_rm_score(rm_model, tokenizer, text):
    input_ids = tokenizer.encode(text, return_tensors='pt',
                                 truncation=True, max_length=256).to(device)
    with torch.no_grad():
        score = rm_model(input_ids)
    return score.cpu().item()

print("=" * 80)
print("[정성+RM점수 평가] SFT 모델 생성 답변의 RM 보상 점수")
print("=" * 80)
results = []
for raw_p in eval_prompts_rm:
    p = PROMPT_DICT['prompt_input'].format_map({'prompt': raw_p})
    sft_text = sft_gen_local(p, **gen_kw)[0]['generated_text']
    sft_ans  = sft_text[len(p):].strip()
    rm_s     = get_rm_score(model, tokenizer, p + sft_ans)
    results.append({'prompt': raw_p, 'sft_answer': sft_ans[:100], 'rm_score': round(rm_s, 4)})
    print(f"\n[PROMPT] {raw_p}")
    print(f"  [SFT 답변] {sft_ans[:120]}")
    print(f"  [RM 점수]  {rm_s:.4f}")

import pandas as pd
df_rm_eval = pd.DataFrame(results)
print("\n[RM 점수 요약]")
print(df_rm_eval[['prompt','rm_score']].to_string(index=False))

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'top_k', 'max_new_tokens', 'early_stopping', 'do_sample', 'eos_token_id', 'no_repeat_ngram_size', 'repetition_penalty'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[정성+RM점수 평가] SFT 모델 생성 답변의 RM 보상 점수


Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[PROMPT] 인공지능(AI)은 무엇인가요?
  [SFT 답변] '저는 인공 지능 AI는 존재하지 않습니다. 인간의 뇌는 인간과 동일한 기능을 가지고 있지만, 다른 사람의 명령에 따라 행동하고 있습니다. 따라서 인간이 명령을 내리지 않거나 이행하지 않을 수 있다는 뜻입니다.提督,
  [RM 점수]  -1.8840


Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[PROMPT] 딥러닝과 머신러닝의 차이를 설명해줘.
  [SFT 답변] '1. AI: 인간의 경험은 인공지능이 개발해낸 것으로, 인간과 다르게 이해할 수 있습니다.\n2. 컴퓨터 모델에서 작동하는 데이터는 인간보다 훨씬 정확합니다. 하지만 이러한 차이는 인간이 학습한 데이터를 기반으로 
  [RM 점수]  -1.9826

[PROMPT] 한국의 수도는 어디인가요?
  [SFT 답변] '한국 수도권에 위치한 서울시 종로구 종로3가 41번지입니다.辛世應(李)으로 불리며, 대한민국 서울특별시로 불리고 있습니다. 현재는 경복궁으로 사용되고 있으며, 주변에 한국민속촌, 한글관, 남산타워, 명소 등을 포함
  [RM 점수]  -2.0876

[RM 점수 요약]
              prompt  rm_score
    인공지능(AI)은 무엇인가요?   -1.8840
딥러닝과 머신러닝의 차이를 설명해줘.   -1.9826
      한국의 수도는 어디인가요?   -2.0876


## 5. Proximal Policy Optimization (PPO)

In [56]:
with NaiveStrategy().model_init_context():
    actor  = GPTActor(pretrained=cfg.output_dir_SFT, lora_rank=0).to(torch.cuda.current_device())
    critic = GPTCritic(pretrained=cfg.output_dir_RM,  lora_rank=0).to(torch.cuda.current_device())
    tokenizer = PreTrainedTokenizerFast.from_pretrained(
        cfg.model_name,
        bos_token=cfg.bos_token, eos_token=cfg.eos_token,
        unk_token=cfg.unk_token, pad_token=cfg.pad_token,
        padding_side=cfg.padding_side,
        model_max_length=cfg.model_max_length,
    )
    initial_model = deepcopy(actor)
    reward_model  = RewardModel(deepcopy(critic.model), deepcopy(critic.value_head)).to(torch.cuda.current_device())

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [57]:
actor_optim  = torch.optim.Adam(actor.parameters(),  lr=cfg.ppo_actor_lr)
critic_optim = torch.optim.Adam(critic.parameters(), lr=cfg.ppo_critic_lr)

In [58]:
(actor, actor_optim), (critic, critic_optim), reward_model, initial_model = NaiveStrategy().prepare(
    (actor, actor_optim), (critic, critic_optim), reward_model, initial_model)

In [59]:
with open(cfg.data_path_3_PPO, "r", encoding='utf-8-sig') as json_file:
    list_data_dict = json.load(json_file)
    list_prompt = [tmp['prompt'] for tmp in list_data_dict]

def tokenize_fn(texts):
    batch = tokenizer(
        texts,
        return_tensors='pt',
        max_length=cfg.ppo_tokenize_max_length,
        padding=True,
        truncation=True,
    )
    return {k: v.cuda() for k, v in batch.items()}

In [62]:
trainer = PPOTrainer(
    NaiveStrategy(),
    actor,
    critic,
    reward_model,
    initial_model,
    actor_optim,
    critic_optim,
    max_epochs=cfg.ppo_max_epochs,
    train_batch_size=cfg.ppo_train_batch_size,
    tokenizer=tokenize_fn,
    max_length=cfg.ppo_max_length,
    do_sample=True,
    temperature=cfg.ppo_temperature,
    top_k=cfg.ppo_top_k,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

In [63]:
trainer.fit(
    list_prompt,
    num_episodes=cfg.ppo_num_episodes,
    max_timesteps=cfg.ppo_max_timesteps,
    update_timesteps=cfg.ppo_update_timesteps,
)
actor.model.save_pretrained(cfg.output_dir_PPO)

Episode [10/10]: 100%|██████████| 3/3 [00:20<00:00,  6.70s/it]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## 6. 최종 모델 비교

In [64]:
# ── Base / SFT / PPO 모델 로드 ──────────────────────────────────────────
tok = PreTrainedTokenizerFast.from_pretrained(
    cfg.model_name,
    bos_token=cfg.bos_token, eos_token=cfg.eos_token,
    unk_token=cfg.unk_token, pad_token=cfg.pad_token,
)

_pipe_kwargs = dict(tokenizer=tok, device=0 if torch.cuda.is_available() else -1)

base_pipe = transformers.pipeline('text-generation', model=cfg.model_name,     **_pipe_kwargs)
sft_pipe  = transformers.pipeline('text-generation', model=cfg.output_dir_SFT, **_pipe_kwargs)
ppo_pipe  = transformers.pipeline('text-generation', model=cfg.output_dir_PPO, **_pipe_kwargs)

PROMPT_DICT = {"prompt_input": "### Instruction(명령어):\n{prompt}\n\n### Response(응답):"}

final_test_prompts = [
    '불고기용 고기 한우에요?',
    '리처드 닉슨이 43대 부통령직을 수행한 년도는?',
    '시카고 오헤어 국제공항은 어디에 있어?',
    '인공지능이 미래 사회에 미치는 영향은 무엇인가?',
    '지구 온난화의 주요 원인은 무엇인가?',
]

gen_kw_final = dict(
    max_new_tokens=cfg.gen_max_new_tokens,
    do_sample=True,
    top_k=cfg.gen_top_k,
    num_beams=cfg.gen_num_beams,
    repetition_penalty=cfg.gen_repetition_penalty,
    no_repeat_ngram_size=cfg.gen_no_repeat_ngram_size,
    eos_token_id=cfg.gen_eos_token_id,
    early_stopping=True,
)

print("=" * 80)
print("[정성 평가] Base KoGPT2 vs SFT vs PPO 최종 비교")
print("=" * 80)
for raw_p in final_test_prompts:
    p = PROMPT_DICT['prompt_input'].format_map({'prompt': raw_p})
    b = base_pipe(p, **gen_kw_final)[0]['generated_text'][len(p):]
    s = sft_pipe(p,  **gen_kw_final)[0]['generated_text'][len(p):]
    o = ppo_pipe(p,  **gen_kw_final)[0]['generated_text'][len(p):]
    print(f"\n[PROMPT] {raw_p}")
    print(f"  [Base] {b.strip()[:120]}")
    print(f"  [SFT]  {s.strip()[:120]}")
    print(f"  [PPO]  {o.strip()[:120]}")

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

...


[PROMPT] 시카고 오헤어 국제공항은 어디에 있어?
  [Base] #selfie #daily #instagood #ootd #like4like #fff #f4f #l4l #lfl #follow #love #good #
  [SFT]  '저는 인공지능 어시스턴트이기 때문에 시카고에 대한 정보를 가지고 있지 않습니다. 하지만 시카고는 미국 캘리포니아주 로스앤젤레스(LA) 지역에 위치해 있습니다.證據, physical responses of this 
  [PPO]  '저는 인공지능 어시스턴트이기 때문에 시카고에 대한 정보를 가지고 있지 않습니다. 하지만 시카고는 미국 캘리포니아주 로스앤젤레스(LA) 지역에 위치해 있습니다. 시카고는 미국에서 가장 큰 도시 중 하나입니다. 시카고


Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[PROMPT] 인공지능이 미래 사회에 미치는 영향은 무엇인가?
  [Base] 
  [SFT]  '인공지능은 미래 사회에 미치는 영향을 예측하고 예측하는 능력을 가지고 있습니다. 인공지능은 인간의 행동과 감정을 예측하고 예측할 수 있는 능력을 가지고 있습니다. 이러한 능력은 인문학, 사회과학, 예술 등 다양한 
  [PPO]  '인공지능의 미래 사회에 미치는 영향으로는 다음과 같은 것들이 있을 수 있습니다.\n\n1. 인공지능 기술의 발전: 인공지능 기술은 인간의 삶을 더욱 풍요롭게 만들어줍니다. 이를 통해 인간 삶의 질이 향상되고, 더 


Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[PROMPT] 지구 온난화의 주요 원인은 무엇인가?
  [Base] 
  [SFT]  '저는 인공지능 어시스턴트이기 때문에 지구 온난화에 대한 구체적인 정보를 가지고 있지 않습니다. 하지만 지구 온난화는 지구 온난화와 밀접한 관련이 있습니다. 지구 온난화는 기후 변화로 인해 발생하는 것으로 알려져 있
  [PPO]  '저는 인공지능 어시스턴트이기 때문에 지구 온난화에 대한 정확한 답변을 제공할 수 없습니다. 하지만 일반적으로 지구 온난화는 기후 변화로 인해 발생하는 것으로 알려져 있습니다. 지구 온난화는 지구 온난화를 유발하는 


In [65]:
# ── 정량 평가: BLEU / ROUGE / Perplexity (Base vs SFT vs PPO) ───────────
import evaluate, json, random, torch
import pandas as pd
from transformers import AutoModelForCausalLM, PreTrainedTokenizerFast

bleu_m  = evaluate.load("bleu")
rouge_m = evaluate.load("rouge")

# 평가 데이터 샘플
import os
sft_eval_path = (cfg.data_path_1_SFT_clean
                 if os.path.exists(cfg.data_path_1_SFT_clean)
                 else cfg.data_path_1_SFT)
with open(sft_eval_path, 'r', encoding='utf-8-sig') as f:
    eval_data = json.load(f)
random.seed(42)
sample = random.sample(eval_data, min(100, len(eval_data)))

PROMPT_DICT = {"prompt_input": "### Instruction(명령어):\n{prompt}\n\n### Response(응답):"}
prompts_q   = [PROMPT_DICT['prompt_input'].format_map({'prompt': d['prompt']}) for d in sample]
refs_text_q = [d['completion'] for d in sample]
refs_bleu_q = [[r] for r in refs_text_q]

gen_kw_q = dict(max_new_tokens=64, do_sample=False, num_beams=4,
                repetition_penalty=2.0, no_repeat_ngram_size=4,
                eos_token_id=cfg.gen_eos_token_id, early_stopping=True)

def gen_list_q(pipe, prompts):
    outs = []
    for i in range(0, len(prompts), 8):
        batch = pipe(prompts[i:i+8], **gen_kw_q)
        for j, res in enumerate(batch):
            text = res[0]['generated_text']
            outs.append(text[len(prompts[i+j]):].strip())
    return outs

print("Base 생성 중...")
base_preds_q = gen_list_q(base_pipe, prompts_q)
print("SFT 생성 중...")
sft_preds_q  = gen_list_q(sft_pipe,  prompts_q)
print("PPO 생성 중...")
ppo_preds_q  = gen_list_q(ppo_pipe,  prompts_q)

def safe_bleu(preds, refs):
    return bleu_m.compute(predictions=preds, references=refs)['bleu']

def safe_rouge(preds, refs):
    return rouge_m.compute(predictions=preds, references=refs)['rougeL']

# Perplexity
def calc_ppl(model_path, texts):
    tok = PreTrainedTokenizerFast.from_pretrained(cfg.model_name)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    mdl = AutoModelForCausalLM.from_pretrained(model_path).to(device)
    mdl.eval()
    ppls = []
    for t in texts[:20]:
        enc = tok(t, return_tensors='pt', truncation=True, max_length=256).to(device)
        with torch.no_grad():
            loss = mdl(**enc, labels=enc['input_ids']).loss
        ppls.append(torch.exp(loss).item())
    del mdl; torch.cuda.empty_cache()
    return sum(ppls)/len(ppls)

print("PPL 계산 중...")
ppl_b = calc_ppl(cfg.model_name,   refs_text_q)
ppl_s = calc_ppl(cfg.output_dir_SFT, refs_text_q)
ppl_p = calc_ppl(cfg.output_dir_PPO, refs_text_q)

df_final = pd.DataFrame({
    'Model':        ['Base KoGPT2', 'SFT', 'PPO'],
    'BLEU':         [round(safe_bleu(base_preds_q, refs_bleu_q), 4),
                     round(safe_bleu(sft_preds_q,  refs_bleu_q), 4),
                     round(safe_bleu(ppo_preds_q,  refs_bleu_q), 4)],
    'ROUGE-L':      [round(safe_rouge(base_preds_q, refs_text_q), 4),
                     round(safe_rouge(sft_preds_q,  refs_text_q), 4),
                     round(safe_rouge(ppo_preds_q,  refs_text_q), 4)],
    'Perplexity ↓': [round(ppl_b,2), round(ppl_s,2), round(ppl_p,2)],
})

print("\n[최종 정량 평가 결과] Base vs SFT vs PPO")
print(df_final.to_string(index=False))

Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Base 생성 중...


Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `ma

SFT 생성 중...


Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `ma

...

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning



[최종 정량 평가 결과] Base vs SFT vs PPO
      Model   BLEU  ROUGE-L  Perplexity ↓
Base KoGPT2 0.0000   0.0005        350.48
        SFT 0.0329   0.0607        459.91
        PPO 0.0343   0.0384        480.15


In [66]:
# ── RM 점수 비교: SFT vs PPO ────────────────────────────────────────────
import torch

def rm_score_final(text):
    input_ids = tokenizer.encode(text, return_tensors='pt',
                                 truncation=True, max_length=256).cuda()
    with torch.no_grad():
        score = reward_model(input_ids)
    return score.cpu().item()

PROMPT_DICT = {"prompt_input": "### Instruction(명령어):\n{prompt}\n\n### Response(응답):"}

score_prompts = [
    '인공지능이 사회에 미치는 영향은?',
    '딥러닝과 머신러닝의 차이를 설명해줘.',
    '지구 온난화의 주요 원인은 무엇인가?',
]

gen_kw_rm = dict(max_new_tokens=64, do_sample=True, top_k=50,
                 repetition_penalty=2.0, no_repeat_ngram_size=4,
                 eos_token_id=cfg.gen_eos_token_id, early_stopping=True)

rows_rm = []
print("[RM 점수 비교] SFT vs PPO")
for raw_p in score_prompts:
    p = PROMPT_DICT['prompt_input'].format_map({'prompt': raw_p})
    sft_ans = sft_pipe(p, **gen_kw_rm)[0]['generated_text'][len(p):]
    ppo_ans = ppo_pipe(p, **gen_kw_rm)[0]['generated_text'][len(p):]
    sft_score = rm_score_final(p + sft_ans)
    ppo_score = rm_score_final(p + ppo_ans)
    rows_rm.append({'prompt': raw_p, 'sft_rm_score': round(sft_score,4),
                    'ppo_rm_score': round(ppo_score,4)})
    print(f"\n[PROMPT] {raw_p}")
    print(f"  SFT RM score: {sft_score:.4f}  |  PPO RM score: {ppo_score:.4f}")

import pandas as pd
df_rm_compare = pd.DataFrame(rows_rm)
print("\n[RM 점수 요약 테이블]")
print(df_rm_compare.to_string(index=False))

Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[RM 점수 비교] SFT vs PPO


Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[PROMPT] 인공지능이 사회에 미치는 영향은?
  SFT RM score: -0.3458  |  PPO RM score: -0.3491


Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[PROMPT] 딥러닝과 머신러닝의 차이를 설명해줘.
  SFT RM score: -0.3411  |  PPO RM score: -0.3337


Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[PROMPT] 지구 온난화의 주요 원인은 무엇인가?
  SFT RM score: -0.3723  |  PPO RM score: -0.3652

[RM 점수 요약 테이블]
              prompt  sft_rm_score  ppo_rm_score
  인공지능이 사회에 미치는 영향은?       -0.3458       -0.3491
딥러닝과 머신러닝의 차이를 설명해줘.       -0.3411       -0.3337
지구 온난화의 주요 원인은 무엇인가?       -0.3723       -0.3652


### Generation 전략 실험

In [67]:
# ── 디코딩 전략 실험 ────────────────────────────────────────────────────
PROMPT_DICT = {"prompt_input": "### Instruction(명령어):\n{prompt}\n\n### Response(응답):"}
test_p = PROMPT_DICT['prompt_input'].format_map({'prompt': '인공지능이 미래 사회에 미치는 영향은?'})

strategies = {
    "Greedy":         dict(max_new_tokens=80, do_sample=False),
    "Beam-4":         dict(max_new_tokens=80, do_sample=False, num_beams=4,
                           no_repeat_ngram_size=4, repetition_penalty=2.0,
                           eos_token_id=cfg.gen_eos_token_id, early_stopping=True),
    "Top-k(k=50)":    dict(max_new_tokens=80, do_sample=True, top_k=50),
    "Top-p(p=0.9)":   dict(max_new_tokens=80, do_sample=True, top_p=0.9),
    "Beam+Top-k":     dict(max_new_tokens=80, do_sample=True, num_beams=4,
                           top_k=50, repetition_penalty=2.0,
                           no_repeat_ngram_size=4, early_stopping=True),
}

print("=" * 80)
print("[생성 전략 비교] PPO 모델 기준")
print("=" * 80)
rows_strat = []
for name, kw in strategies.items():
    out = ppo_pipe(test_p, **kw)[0]['generated_text'][len(test_p):]
    rows_strat.append({'strategy': name, 'output': out.strip()[:150]})
    print(f"\n[{name}]\n{out.strip()[:200]}")

df_strategies = pd.DataFrame(rows_strat)

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[생성 전략 비교] PPO 모델 기준


Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[Greedy]
'인공지능이 미래 사회에 미치는 영향은 다양합니다. 예를 들어, 인공지능은 인간의 지능과 관련된 다양한 문제를 해결하고 해결책을 제시할 수 있습니다. 이러한 문제들은 인간의 성장과 발전에 큰 영향을 미칩니다. 이러한 문제들은 인간의 성장과 발전에 큰 영향을 미칩니다. 이러한 문제들은 인간의 성장과 발전에 큰 영향을 미칩니다.


Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'top_k'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[Beam-4]
'인공지능은 미래에 발생할 수 있는 여러 가지 요인에 의해 영향을 받는다고 알려져 있습니다. 예를 들어, 인공지능은 인간, 동물, 식물, 미생물 등 다양한 요인에 의해 영향을 받습니다. 이러한 요인들은 인간의 삶에서 중요한 요소 중 하나이기 때문에, 인간과 동물 모두에게 큰 영향을 미칠 수 있습니다. 또한, 인공지능 기술은 인간의 삶에 대한 이해와 이해를 


Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'top_p'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[Top-k(k=50)]
'인공지능은 미래에 발생할 수 있는 문제에 대한 예측능력, 정보 접근성, 보안 등 다양한 분야에서 영향을 미치며, 모든 인간의 삶에 영향을 미치는 것은 아니므로, 이에 대한 정확한 예측은 가능하지 않습니다. 하지만, 인공지능이 미래에 발생하는 문제와 관련된 예측 능력을 발전시키기 위해서는 인간의 지식과 경험, 그리고 사회적 역할에 대한 이해와 노력이 필요합


Passing `generation_config` together with generation-related arguments=({'top_k', 'max_new_tokens', 'num_beams', 'early_stopping', 'do_sample', 'no_repeat_ngram_size', 'repetition_penalty'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[Top-p(p=0.9)]
'인공지능은 미래 사회의 변화를 선도하고 사회의 발전과 발전에 큰 역할을 합니다. 또한, 인간의 성장과 발전에 영향을 미치는 중요한 요소 중 하나입니다. 이러한 변화는 인간 개개인의 성장과 발전을 촉진하는 역할을 합니다. 따라서 인공지능은 인간의 성장과 발전에 큰 역할을 합니다. 따라서 인공지능은 인간의 성장과 발전을 촉진하는 역할을 합니다.

[Beam+Top-k]
'저는 인공지능 어시스턴트이기 때문에 인공지능이 미래에 미치는 영향을 알 수 없습니다. 하지만, 인공지능은 인간의 창의성과 문제해결 능력을 높이는 데 큰 역할을 합니다.
